# Block 2 (Archived): Three-Season Dirichlet Weighting for Bayesian MARCEL (wRC+)

**This notebook is archived. Do not refit the model.**

The fitted trace is saved at `archive/block2_trace.pkl`. This notebook documents
the three-season specification and can be used to reload and inspect the archived trace.

---

## Context

This is the initial Block 2 implementation, following the PyMC Labs / Tango MARCEL convention
of using three prior seasons with a Dirichlet-distributed weight vector.

The **primary model** is the two-season specification in `../marcel_block2.ipynb` and
`src/models/marcel.py`. The three-season model is retained here as a robustness reference
and for the writeup, which will compare the two approaches.

### Why the three-season model was archived

Two reasons (both documented in `claude_instructions/wrc_modeling_decisions.md`):

1. **Comparative framework alignment.** The regression baselines (linear, LASSO) use a
   two-season input window. A three-season Bayesian model vs. two-season frequentist
   baselines would confound model effect with data scope.
2. **Weight non-monotonicity.** The three-season fit (target years 2019–2023) produced
   posterior weights that violated the expected recency ordering at the middle position
   (w[t-2] < w[t-3]). Diagnostic investigation traced this to multi-target-year pooling;
   the two-season specification avoids this by construction.

### Three-season posterior weights (archived result)

- w[t-3] = 0.289
- w[t-2] = 0.163  ← non-monotonic
- w[t-1] = 0.549

P(w[t-3] > w[t-2]) ≈ 94.9% in the posterior.

In [ ]:
import sys
sys.path.insert(0, '../../../../src')

import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

# Block 1 data loading from marcel.py (unchanged between two- and three-season models)
from models.marcel import load_and_filter, run_sanity_checks

## Three-season model code (inline)

The three-season functions below were removed from `src/models/marcel.py` when the
two-season specification became primary. They are reproduced here for reference.

In [ ]:
# Target years for the three-season model: requires t-1, t-2, t-3 or partial history.
# Earliest possible target is 2019 (requires 2016 as t-3).
PREDICTION_SEASONS_3S = list(range(2019, 2024))
MIN_PA = 30


def build_prediction_table_3season(df: pd.DataFrame) -> pd.DataFrame:
    """Build the three-season prediction input table.

    For each (player, target_season) in 2019–2023, look up the player's prior
    1–3 seasons in df and record positional indices into df's row order
    (which matches the theta vector in Block 1).
    """
    idx_map = {
        (row.IDfg, row.Season): i
        for i, row in df.iterrows()
    }

    rows = []
    for target_season in PREDICTION_SEASONS_3S:
        targets = df[df["Season"] == target_season]
        for _, tgt in targets.iterrows():
            pid = tgt["IDfg"]

            idx_t1 = idx_map.get((pid, target_season - 1), -1)
            idx_t2 = idx_map.get((pid, target_season - 2), -1)
            idx_t3 = idx_map.get((pid, target_season - 3), -1)

            n_prior = sum(1 for idx in [idx_t1, idx_t2, idx_t3] if idx != -1)
            if n_prior == 0:
                continue

            rows.append({
                "IDfg": pid,
                "Name": tgt["Name"],
                "Season": target_season,
                "wRC+": tgt["wRC+"],
                "PA": tgt["PA"],
                "theta_idx_t1": idx_t1,
                "theta_idx_t2": idx_t2,
                "theta_idx_t3": idx_t3,
                "n_prior": n_prior,
            })

    pred_df = pd.DataFrame(rows).reset_index(drop=True)
    pred_df["pred_key"] = (
        pred_df["IDfg"].astype(str) + "_" + pred_df["Season"].astype(str)
    )
    return pred_df


def run_prediction_sanity_checks_3season(pred_df: pd.DataFrame) -> None:
    """Assert three-season prediction table quality requirements."""
    assert pred_df["PA"].min() >= MIN_PA
    assert set(pred_df["Season"].unique()).issubset(set(PREDICTION_SEASONS_3S))
    assert set(pred_df["n_prior"].unique()).issubset({1, 2, 3})

    for _, row in pred_df.iterrows():
        n_valid = sum(1 for c in ["theta_idx_t1", "theta_idx_t2", "theta_idx_t3"]
                      if row[c] != -1)
        assert n_valid == row["n_prior"], (
            f"n_prior mismatch for {row['IDfg']}_{row['Season']}: "
            f"n_prior={row['n_prior']} but {n_valid} valid indices"
        )

    print(f"Prediction table checks passed. {len(pred_df)} rows, "
          f"n_prior distribution: {pred_df['n_prior'].value_counts().sort_index().to_dict()}")


def build_block2_3season_model(df: pd.DataFrame, pred_df: pd.DataFrame) -> pm.Model:
    """Construct the joint Block 1 + Block 2 three-season model.

    Block 1: hierarchical pooling (rate_like).
    Block 2: Dirichlet(3, 4, 5) season weighting over three prior seasons
             (prediction likelihood).
    """
    wrc_obs = df["wRC+"].values.astype(float)
    pa_obs = df["PA"].values.astype(float)
    coords = {
        "player_season": df["player_season"].values,
        "pred_row": pred_df["pred_key"].values,
        "w_season": ["t-3", "t-2", "t-1"],
    }

    target_wrc = pred_df["wRC+"].values.astype(float)
    target_pa = pred_df["PA"].values.astype(float)

    idx_t1 = pred_df["theta_idx_t1"].values.astype(int)
    idx_t2 = pred_df["theta_idx_t2"].values.astype(int)
    idx_t3 = pred_df["theta_idx_t3"].values.astype(int)

    mask_t1 = (idx_t1 != -1).astype(float)
    mask_t2 = (idx_t2 != -1).astype(float)
    mask_t3 = (idx_t3 != -1).astype(float)

    # Clamp -1 indices to 0 so gather doesn't fail; masked out by weight logic
    safe_t1 = np.where(idx_t1 == -1, 0, idx_t1)
    safe_t2 = np.where(idx_t2 == -1, 0, idx_t2)
    safe_t3 = np.where(idx_t3 == -1, 0, idx_t3)

    with pm.Model(coords=coords) as model:
        # === Block 1: rate_like ===
        wrc_data = pm.MutableData("wrc_data", wrc_obs)
        pa_data = pm.MutableData("pa_data", pa_obs)

        mu_theta = pm.Normal("mu_theta", mu=100, sigma=5)
        sigma_theta = pm.HalfNormal("sigma_theta", sigma=20)
        sigma_obs = pm.HalfNormal("sigma_obs", sigma=500)

        theta_raw = pm.Normal("theta_raw", mu=0, sigma=1, dims="player_season")
        theta = pm.Deterministic(
            "theta", mu_theta + sigma_theta * theta_raw, dims="player_season"
        )

        obs_sd = sigma_obs / pm.math.sqrt(pa_data)
        pm.Normal("wrc_plus", mu=theta, sigma=obs_sd, observed=wrc_data,
                  dims="player_season")

        # === Block 2: three-season Dirichlet weighting ===
        # Dirichlet(3, 4, 5): expected weights (0.25, 0.33, 0.42), encoding Tango's
        # 5/4/3 recency scheme. w[0]=t-3, w[1]=t-2, w[2]=t-1.
        w = pm.Dirichlet("w", a=np.array([3.0, 4.0, 5.0]), dims="w_season")

        target_wrc_data = pm.MutableData("target_wrc_data", target_wrc)
        target_pa_data = pm.MutableData("target_pa_data", target_pa)
        mask_t1_data = pm.MutableData("mask_t1_data", mask_t1)
        mask_t2_data = pm.MutableData("mask_t2_data", mask_t2)
        mask_t3_data = pm.MutableData("mask_t3_data", mask_t3)
        safe_t1_data = pm.MutableData("safe_t1_data", safe_t1)
        safe_t2_data = pm.MutableData("safe_t2_data", safe_t2)
        safe_t3_data = pm.MutableData("safe_t3_data", safe_t3)

        theta_t1 = theta[safe_t1_data]
        theta_t2 = theta[safe_t2_data]
        theta_t3 = theta[safe_t3_data]

        # Positional weights: w[2]=t-1 (most recent), w[1]=t-2, w[0]=t-3
        raw_w_t1 = w[2] * mask_t1_data
        raw_w_t2 = w[1] * mask_t2_data
        raw_w_t3 = w[0] * mask_t3_data

        w_sum = raw_w_t1 + raw_w_t2 + raw_w_t3
        norm_w_t1 = raw_w_t1 / w_sum
        norm_w_t2 = raw_w_t2 / w_sum
        norm_w_t3 = raw_w_t3 / w_sum

        theta_proj = pm.Deterministic(
            "theta_proj",
            norm_w_t1 * theta_t1 + norm_w_t2 * theta_t2 + norm_w_t3 * theta_t3,
            dims="pred_row",
        )

        pred_sd = sigma_obs / pm.math.sqrt(target_pa_data)
        pm.Normal("wrc_pred", mu=theta_proj, sigma=pred_sd,
                  observed=target_wrc_data, dims="pred_row")

    return model

## 1. Load data

In [ ]:
df = load_and_filter('../../../../data/fg_builds/fg_hitters_train.csv')
run_sanity_checks(df)
print(f"\nBlock 1 table: {len(df)} rows, {df['IDfg'].nunique()} players")

## 2. Build prediction table (three-season)

In [ ]:
pred_df = build_prediction_table_3season(df)
run_prediction_sanity_checks_3season(pred_df)
print(f"\nPrediction table: {len(pred_df)} rows")
print(f"n_prior distribution:\n{pred_df['n_prior'].value_counts().sort_index()}")

## 3. Build model (reference only — do not refit)

The model object is constructed below for documentation purposes and to allow
loading the trace into context. The model was fit on 2016–2023 training data
with tune=2000, draws=1000, target_accept=0.95.

The fitted trace is at `archive/block2_trace.pkl`.

In [ ]:
model = build_block2_3season_model(df, pred_df)

## 4. Load archived trace and inspect results

In [ ]:
import pickle

with open('block2_trace.pkl', 'rb') as f:
    trace = pickle.load(f)
print("Trace loaded from archive/block2_trace.pkl")

In [ ]:
# Posterior summary for weights
w_summary = az.summary(trace, var_names=["w"])
print("Posterior summary (weights):")
print(w_summary)

w_means = trace.posterior["w"].mean(dim=["chain", "draw"]).values
print(f"\nw posterior means: t-3={w_means[0]:.3f}, t-2={w_means[1]:.3f}, t-1={w_means[2]:.3f}")
print(f"Expected monotonic ordering (t-3 < t-2 < t-1): {w_means[0] < w_means[1] < w_means[2]}")

In [ ]:
# Fraction of posterior samples where w[t-3] > w[t-2] (the non-monotonicity)
samples = trace.posterior["w"].values  # shape: (chains, draws, 3)
samples_flat = samples.reshape(-1, 3)
frac = (samples_flat[:, 0] > samples_flat[:, 1]).mean()
print(f"P(w[t-3] > w[t-2]) = {frac:.2%}")
print("(~95% in the archived fit — see wrc_modeling_decisions.md for diagnostic investigation)")

In [ ]:
# Hyper-parameter summary
hyper_summary = az.summary(trace, var_names=["mu_theta", "sigma_theta", "sigma_obs"])
print("Posterior summary (hyper-parameters):")
print(hyper_summary)